## 20. 🆕 Backtest de insumos y estabilidad temporal

**Problema.** La sección 16 explicó que el $TBP_C$ es una *identidad contable* y que no corresponde una regresión. Otra parte del proyecto agregó un test "in-sample / out-of-sample" (cohortes 1950-2010 vs 2011-2024, caída de −28,6 %, Chow con $p=0{,}0005$). Esta sección **evalúa críticamente ese test** y hace lo que **sí** se puede validar:

* **(A) Backtest de insumos:** comparar proyecciones *pasadas* de los insumos ($N$ nacimientos, $J$ población 65+) con lo que efectivamente ocurrió. Eso sí es "fuera de muestra" de verdad.
* **(B) Estabilidad temporal:** ver si el "motor" de la serie cambia entre tramos y dónde cambia la pendiente, con las salvedades del caso.
* **(C) Tabla final** de qué sirve validar y qué no (19.10).

No se descarga nada: se usan datos que ya están en el proyecto; lo que falta queda en una lista (19.9).

### 20.1 Por qué el test IS/OOS existente no es lo que parece

1. **No hay resultado observado.** Para validar fuera de muestra hace falta una variable que se observe *después* (p. ej. el déficit previsional) y comparar contra ella. El $TBP_C$ no tiene esa variable: es una cuenta. Además los TBP de las cohortes 2011-2024 usan $J$, $E_r$, $S_{65}$ y $ho$ **proyectados** a 2076-2089. Llamar "OOS" a ese tramo es solo una etiqueta de calendario, no una predicción puesta a prueba.
2. **Los $p$-valores no significan lo habitual.** La serie es determinística, suave y muy autocorrelada: valores vecinos casi se repiten. Un Chow supone errores independientes; acá no hay error aleatorio. Sirve como **descripción** de un cambio de pendiente, no como prueba. (En 19.8 se ve que, con una permutación por bloques, el $p$ del sup-F pasa de 0,0005 a 0,4-0,6.)
3. **El corte 2011 es arbitrario.** Los nacimientos del DEIS tienen su pico en 2014 y la caída arranca en 2015. El estadístico, además, es casi igual para cualquier corte entre 2013 y 2018 (19.8).
4. **El "look-ahead bias" está mal planteado.** Ningún parámetro se ajustó para reproducir una trayectoria de déficit, así que no hay sobreajuste que "mirar hacia adelante". El problema real es el **anacronismo**: se aplican parámetros actuales ($\bar A$, $\rho$, $\tau$ de 2017-2025) a cohortes lejanas y se supone **estabilidad estructural**. Eso no se testea con un corte temporal; se discute con sensibilidad (sección 11 y 16).

**Qué sí es defendible:** decir que las cohortes recientes tienen un $TBP_C$ menor (descripción), y que esa baja viene sobre todo de $N$ (descomposición, 19.7). **Qué no:** hablar de "degradación OOS" o de "quiebre estructural significativo".

### 20.2 Módulo (copia de `src/tbp_estabilidad.py`)

Todo el código está en una sola celda para que el Colab sea autosuficiente. Funciones de backtest (`error_pct`, `backtest_tabla`, `mc_tbp`, `PENDIENTES_DESCARGA`...) y de estabilidad (`componentes_log`, `ventanas_moviles`, `sup_f`, `wald_hac`, `permutacion_bloques`, `tabla_quiebre`). Los datos transcriptos del PDF de INDEC 2025 llevan la página de origen en comentarios.

In [ ]:
# ====== MÓDULO tbp_estabilidad (misma versión que src/tbp_estabilidad.py; se omite el bloque __main__) ======
_autotest_previo_19 = globals().get('autotest')   # el autotest de este módulo no pisa el de otras secciones
# -*- coding: utf-8 -*-
"""
tbp_estabilidad.py — Backtest de insumos y estabilidad temporal del TBP_C.

Proyecto TBP — Índice de Cuna Vacía (Taller de Indicadores).

POR QUÉ ESTE MÓDULO (en una línea): el TBP_C es una IDENTIDAD contable sin variable resultado
observada, así que NO existe un "fuera de muestra" predictivo para el índice. Lo que SÍ se puede
validar es (A) cuánto se equivocaron proyecciones pasadas de los INSUMOS (nacimientos N, población 65+ J)
y (B) si el "motor" de la serie cambia a lo largo del tiempo (estabilidad), con las salvedades del caso.

    TBP_C = N · (1−μ) · Ā · (τ/ρ) / J          ⇒   ln TBP = ln N + ln(1−μ) + ln Ā + ln(τ/ρ) − ln J     (exacta)

CONTENIDO
  A) Backtest de insumos:   error_pct, interpolar_a_fecha, backtest_nacimientos_wpp2024, backtest_poblacion_indec2013,
                            comparar_J_fuentes, backtest_tabla (arma backtest_insumos.csv), PENDIENTES_DESCARGA,
                            sigma_por_horizonte, mc_tbp (Monte Carlo con bandas empíricas en lugar de ±1,5 %).
  B) Estabilidad temporal:  componentes_log, ventanas_moviles, chow_f, sup_f, wald_hac, sup_wald_hac,
                            permutacion_bloques, tabla_quiebre.
  C) autotest() con datos SINTÉTICOS rotulados (no son Argentina).

REGLA: no descarga nada. Todo lo que no está en el proyecto queda en PENDIENTES_DESCARGA (solo lista).
"""
import os, datetime as _dt
import numpy as np
import pandas as pd

RUTAS_CANDIDATAS = ["../datos/procesados", "datos/procesados", "../datos/crudos", "datos/crudos"]

# ----------------------------------------------------------------------------------------------
# Datos transcriptos de PDF locales (INDEC 2025, base Censo 2022). Se citan con página.
# Fuente: datos_tbp/indec_proyecciones_nacionales_2022_2040.pdf  (INDEC, "Estimaciones y proyecciones de
# población, por sexo y edad. Total del país. Años 2022-2040", 1.ª ed., 2025).
# ----------------------------------------------------------------------------------------------
CENSO_2022 = {
    "fecha": _dt.date(2022, 5, 18),
    "total_enumerado": 45_892_285,        # PDF INDEC 2025, p. 12, Tabla 1 (Población 2022)
    "total_sin_calle": 45_886_580,        # PDF p. 19, Tabla 4 (no incluye situación de calle sin edad)
    "j65_censada": 5_464_057,             # PDF p. 19, Tabla 4 (65 y más, ambos sexos, censada)
    "total_estimada_a_fecha_censal": 46_122_853,   # PDF p. 19, Tabla 4 (estimación INDEC a la fecha censal)
    "j65_estimada_a_fecha_censal": 5_521_560,      # PDF p. 19, Tabla 4
}
CENSO_2010 = {"fecha": _dt.date(2010, 10, 27), "total_enumerado": 40_117_096}  # PDF 2025, p. 12, Tabla 1

# Población al 1-jul de cada año (INDEC 2025). Total: fila "Total" del Cuadro 2.1 (pp. 39 y 41).
# 65+: suma de las edades 65..99 y "100 y más" del Cuadro 2.1 (pp. 39-42), calculada al transcribir (la suma de
# todas las edades reproduce el Total sin diferencias).
INDEC_2025_TOTAL = {2022: 46135579, 2023: 46214462, 2024: 46301743, 2025: 46387098, 2026: 46466688, 2027: 46540071,
                    2028: 46609533, 2029: 46676999, 2030: 46744389, 2031: 46814264, 2032: 46886737, 2033: 46961803,
                    2034: 47039274, 2035: 47118716, 2036: 47199436, 2037: 47280504, 2038: 47360804, 2039: 47439108,
                    2040: 47514116}
INDEC_2025_J65 = {2022: 5529167, 2023: 5610238, 2024: 5711979, 2025: 5815927, 2026: 5922327, 2027: 6031328,
                  2028: 6142648, 2029: 6255112, 2030: 6367718, 2031: 6480544, 2032: 6594032, 2033: 6709837,
                  2034: 6831713, 2035: 6963335, 2036: 7105814, 2037: 7258199, 2038: 7421088, 2039: 7597415,
                  2040: 7789606}
INDEC_2025_EDAD0 = {2022: 499587, 2023: 471623, 2024: 453124, 2025: 444076}   # población de 0 años al 1-jul (p. 39)

# Archivos que HABRÍA QUE BAJAR (solo lista; tamaños medidos con cabeceras HTTP HEAD, sin descargar).
PENDIENTES_DESCARGA = [
    dict(archivo="INDEC 2013 — Serie Análisis Demográfico n.º 35 (PDF)",
         url="https://www.indec.gob.ar/ftp/cuadros/publicaciones/proyeccionesyestimaciones_nac_2010_2040.pdf",
         bytes=267_025, para="65+ proyectado para 2020 (Cuadro 2, por sexo y edad quinquenal) y TGF/natalidad (Cuadro 5) → "
         "error de J y de N de la proyección 2013 contra Censo 2022 y DEIS. Verificado: HTTP 200, application/pdf."),
    dict(archivo="WPP 2019 (paquete R 'wpp2019_1.1-1.tar.gz', datos de la ONU)",
         url="https://cran.r-project.org/src/contrib/wpp2019_1.1-1.tar.gz", bytes=4_684_504,
         para="Nacimientos/fecundidad y población por edad proyectados en 2019 → error a 4-5 años (2023-24) y 65+ 2022. "
              "Contenido exacto de tablas: confirmar al abrirlo."),
    dict(archivo="WPP 2017 (paquete R 'wpp2017_1.2-3.tar.gz')",
         url="https://cran.r-project.org/src/contrib/wpp2017_1.2-3.tar.gz", bytes=4_743_882,
         para="Vintage intermedio (horizonte 6-7 años)."),
    dict(archivo="WPP 2015 (paquete R 'wpp2015_1.1-2.tar.gz', carpeta Archive de CRAN)",
         url="https://cran.r-project.org/src/contrib/Archive/wpp2015/wpp2015_1.1-2.tar.gz", bytes=4_585_228,
         para="Horizonte ~8-9 años para 2023-24 y ~7 años para el Censo 2022."),
    dict(archivo="WPP 2012 (paquete R 'wpp2012_2.2-1.tar.gz')",
         url="https://cran.r-project.org/src/contrib/wpp2012_2.2-1.tar.gz", bytes=4_902_348,
         para="Horizonte ~10-12 años; sirve para ver cómo crece el error con el horizonte."),
    dict(archivo="WPP 2010", url="(no verificado: sin paquete en CRAN ni URL vigente hallada; las URL históricas "
         "population.un.org/wpp/Download/Files/... devolvieron 404)", bytes=None,
         para="Habría que pedirlo al archivo de la División de Población de la ONU; no se puede estimar el tamaño."),
]


def tamano_pendientes():
    """Suma (en MB) de lo verificable de PENDIENTES_DESCARGA."""
    return sum(p["bytes"] for p in PENDIENTES_DESCARGA if p["bytes"]) / 1e6


# ==============================================================================================
# A) BACKTEST DE INSUMOS
# ==============================================================================================
def error_pct(proyectado, observado):
    """Error relativo en %: 100·(proyectado/observado − 1). Positivo = la proyección SOBRESTIMÓ."""
    return 100.0 * (np.asarray(proyectado, float) / np.asarray(observado, float) - 1.0)


def interpolar_a_fecha(serie_1jul, fecha):
    """Interpola linealmente una serie al 1-jul de cada año a una fecha calendario (p. ej. 18-may-2022)."""
    f0 = _dt.date(fecha.year if fecha >= _dt.date(fecha.year, 7, 1) else fecha.year - 1, 7, 1)
    f1 = _dt.date(f0.year + 1, 7, 1)
    w = (fecha - f0).days / (f1 - f0).days
    return float(serie_1jul[f0.year] + (serie_1jul[f1.year] - serie_1jul[f0.year]) * w)


def _fila(bloque, insumo, proyeccion, publicada, ref, horizonte, proy, obs, unidad, tipo, fuente_obs, nota):
    return dict(bloque=bloque, insumo=insumo, proyeccion=proyeccion, publicada=publicada, referencia=ref,
                horizonte_anios=horizonte, proyectado=proy, observado=obs, unidad=unidad,
                error_abs=(None if (proy is None or obs is None) else proy - obs),
                error_pct=(None if (proy is None or obs is None) else float(error_pct(proy, obs))),
                tipo=tipo, fuente_observado=fuente_obs, nota=nota)


def backtest_nacimientos_wpp2024(s):
    """ONU WPP 2024 (publicada jul-2024) vs nacimientos DEIS. `s` = series_tbp_extraidas.csv indexado por 'year'
    (un_births_miles en MILES; deis_nacimientos en personas)."""
    filas = []
    for y in (2022, 2023, 2024):
        proy = float(s.un_births_miles[y]) * 1000
        obs = float(s.deis_nacimientos[y])
        extra = ("El error ≈ 0 en 2022 sugiere que ése fue el último año con dato incorporado: 2023 y 2024 son pronósticos a 1 y 2 años."
                 if y == 2022 else "DEIS 2024 puede ser provisorio (inscripciones tardías suben el dato) → el error de 2024 podría ser algo menor.")
        filas.append(_fila("A1 nacimientos", f"Nacimientos {y}", "ONU WPP 2024 (medio)", "jul-2024", y, y - 2022,
                           proy, obs, "personas", "backtest legítimo (proyección publicada antes del dato)",
                           "DEIS (Min. Salud), series_tbp_extraidas.csv", extra))
    return filas


def backtest_poblacion_indec2013(x13, j_wpp_miles=None, tot_wpp_miles=None):
    """INDEC 2013 (base Censo 2010; población total al 1-jul 2010-2040) contra el Censo 2022 y contra la reestimación
    INDEC 2025. `x13`: Series año→población (Cuadro 1 del xls). Devuelve lista de filas."""
    filas = []
    p22 = interpolar_a_fecha(x13, CENSO_2022["fecha"])
    filas.append(_fila("A2 población total", "Población total al 18-may-2022", "INDEC 2013 (base Censo 2010)", "nov-2013",
                       2022, 12, p22, CENSO_2022["total_enumerado"], "personas",
                       "backtest legítimo (12 años)", "Censo 2022 enumerado — PDF INDEC 2025, p. 12 (Tabla 1)",
                       "Proyección interpolada al 18-may-2022 (publicada al 1-jul)."))
    filas.append(_fila("A2 población total", "Población total al 18-may-2022 (vs estimación INDEC a fecha censal)",
                       "INDEC 2013 (base Censo 2010)", "nov-2013", 2022, 12, p22,
                       CENSO_2022["total_estimada_a_fecha_censal"], "personas", "backtest legítimo (12 años)",
                       "INDEC 2025 p. 19 (Tabla 4): estimación a la fecha censal, que corrige por omisión/sobrecobertura",
                       "Comparación más justa: el censo enumerado tiene subnumeración de 0,5 %."))
    p10 = interpolar_a_fecha(x13, CENSO_2010["fecha"])
    filas.append(_fila("A2 población total", "Población total al 27-oct-2010 (año base)", "INDEC 2013 (base Censo 2010)",
                       "nov-2013", 2010, 0, p10, CENSO_2010["total_enumerado"], "personas",
                       "control del año base (no es pronóstico)", "Censo 2010 enumerado — PDF INDEC 2025, p. 12",
                       "La proyección parte del censo CORREGIDO por omisión, por eso queda ~2 % por encima del enumerado."))
    # trayectoria: INDEC 2013 vs reestimación INDEC 2025 (mismo organismo, otra base censal)
    for y in (2025, 2030, 2035, 2040):
        filas.append(_fila("A2 población total", f"Población total {y}: INDEC 2013 vs INDEC 2025", "INDEC 2013", "nov-2013",
                           y, y - 2010, float(x13[y]), float(INDEC_2025_TOTAL[y]), "personas",
                           "revisión entre vintages (la 'realidad' es otra proyección, no un dato)",
                           "INDEC 2025, Cuadro 2.1 (pp. 39-41)",
                           "Muestra cuánto cambia el escenario al incorporar el Censo 2022: mide revisión, no error contra hechos."))
    return filas


def comparar_J_fuentes(j_wpp_miles, tot_wpp_miles=None):
    """65+ de la ONU WPP 2024 (J del TBP) contra (i) Censo 2022 y (ii) INDEC 2025, 2022-2040.
    j_wpp_miles: Series año→65+ en MILES (1-jul)."""
    filas = []
    j22 = interpolar_a_fecha(j_wpp_miles * 1000, CENSO_2022["fecha"])
    filas.append(_fila("A3 población 65+", "65+ al 18-may-2022", "ONU WPP 2024", "jul-2024", 2022, 0, j22,
                       CENSO_2022["j65_censada"], "personas", "chequeo de nivel (WPP 2024 NO incorpora el Censo 2022: ver nota)",
                       "Censo 2022 — PDF INDEC 2025, p. 19 (Tabla 4), censada",
                       "Se verificó que WPP 2024 da 45,41 M para 2022, 1,1 % bajo el censo: no está calibrada con él."))
    filas.append(_fila("A3 población 65+", "65+ al 18-may-2022 (vs estimación INDEC a fecha censal)", "ONU WPP 2024", "jul-2024",
                       2022, 0, j22, CENSO_2022["j65_estimada_a_fecha_censal"], "personas", "chequeo de nivel",
                       "PDF INDEC 2025, p. 19 (Tabla 4)", "El censo subenumera 65+ en 1,0 % (INDEC)."))
    for y in (2022, 2025, 2030, 2035, 2040):
        filas.append(_fila("A3 población 65+", f"65+ {y}: ONU WPP 2024 vs INDEC 2025", "ONU WPP 2024", "jul-2024", y, max(y - 2024, 0),
                           float(j_wpp_miles[y]) * 1000, float(INDEC_2025_J65[y]), "personas",
                           "dispersión entre fuentes (NO es error de proyección)", "INDEC 2025, Cuadro 2.1 (pp. 39-42)",
                           "Sirve de piso para la incertidumbre de J: dos instituciones serias difieren esto."))
    return filas


def dispersion_J(j_wpp_miles):
    """Serie de diferencias % WPP2024 vs INDEC2025 en 65+ (2022-2040) y su RMSE."""
    ys = sorted(INDEC_2025_J65)
    e = pd.Series([float(error_pct(j_wpp_miles[y] * 1000, INDEC_2025_J65[y])) for y in ys], index=ys)
    return e, float(np.sqrt((e ** 2).mean()))


def proxy_edad0(s):
    """Proxy (no es error estricto): población de 0 años al 1-jul-2024 (INDEC 2025) vs promedio de nacimientos DEIS 2023-24
    (los de 0 años a mitad de 2024 nacieron entre jul-2023 y jun-2024)."""
    obs = (float(s.deis_nacimientos[2023]) + float(s.deis_nacimientos[2024])) / 2
    return _fila("A1 nacimientos", "Población de 0 años al 1-jul-2024 vs nacimientos DEIS (prom. 2023-24)", "INDEC 2025", "2025",
                 2024, 0, float(INDEC_2025_EDAD0[2024]), obs, "personas", "proxy (sin corrección por mortalidad infantil/migración)",
                 "DEIS", "INDEC 2025 ve ~3-4 % más nacimientos que el promedio DEIS: coherente con que DEIS 2024 sea provisorio.")


def backtest_tabla(s, x13=None):
    """Arma el DataFrame completo de backtest_insumos.csv (incluye filas 'pendiente de descarga')."""
    filas = backtest_nacimientos_wpp2024(s)
    filas.append(proxy_edad0(s))
    if x13 is not None:
        filas += backtest_poblacion_indec2013(x13)
    filas += comparar_J_fuentes(s.un_pob_65mas_miles)
    for p in PENDIENTES_DESCARGA:
        filas.append(dict(bloque="A4 pendiente", insumo=p["archivo"], proyeccion="(vintage anterior)", publicada="", referencia="",
                          horizonte_anios=None, proyectado=None, observado=None, unidad="", error_abs=None, error_pct=None,
                          tipo="PENDIENTE DE DESCARGA", fuente_observado=p["url"],
                          nota=(f"{p['bytes']/1e6:.2f} MB. " if p["bytes"] else "tamaño desconocido. ") + p["para"]))
    return pd.DataFrame(filas)


def sigma_por_horizonte(sigma_ref, h_ref, h, modo="raiz"):
    """Escala un desvío medido a horizonte h_ref hacia el horizonte h. HEURÍSTICA (supuesto, no estimada):
    'raiz' = el error crece como √h (camino aleatorio); 'constante' = no escala."""
    if modo == "constante" or h_ref <= 0:
        return float(sigma_ref)
    return float(sigma_ref * np.sqrt(max(h, 1) / h_ref))


def mc_tbp(N, S, J, sigma_N=0.015, sigma_J=0.01, n=10000, seed=42, a=(13.5, 15.5, 23.6), tau=(0.183, 0.2335),
           rho=(0.371, 0.403, 0.563)):
    """Monte Carlo idéntico al de la sección 16.4 del notebook (mismos rangos de Ā, τ, ρ y misma semilla y orden de
    sorteos), pero con σ_N y σ_J parametrizables (relativos). Con sigma_N=0,015 y sigma_J=0,01 reproduce el original."""
    np.random.seed(seed)
    Ns = np.random.normal(N, sigma_N * N, n)
    As = np.random.triangular(a[0], a[1], a[2], n)
    ts = np.random.uniform(tau[0], tau[1], n)
    rs = np.random.triangular(rho[0], rho[1], rho[2], n)
    Js = np.random.normal(J, sigma_J * J, n)
    sim = Ns * S * As * ts / (Js * rs)
    return sim, dict(p5=float(np.percentile(sim, 5)), p50=float(np.percentile(sim, 50)), p95=float(np.percentile(sim, 95)),
                     sd=float(sim.std()))


# ==============================================================================================
# B) ESTABILIDAD TEMPORAL
# ==============================================================================================
def componentes_log(av, version="p4", hasta=2024):
    """Descomposición logarítmica EXACTA de TBP_C por cohorte. `av` = tbp_modelo_avanzado_1950_2035.csv.
    ln TBP = ln N + ln S65 + ln Ā + ln(τ/ρ) − ln J.   (S65 = 1−μ).
    version 'p4': N_4, J_star_4 (J* por sexo/edad de retiro), Ā_efectivo = TBP·J*/(N·S·τ/ρ) (incluye el reparto por sexo y
                  edad de retiro: es el Ā que cierra la identidad), ρ del escenario por cohorte.
    version 'p1': N_t, J65 (65+ para todos), Ā = A_eph, τ/ρ constantes (= TBP_p1·J/(N·S·Ā))."""
    a = av.set_index("cohorte") if "cohorte" in av.columns else av
    a = a.loc[:hasta]
    if version == "p4":
        y, N, J, S = a.TBP_C_p4, a.N_4, a.J_star_4, a.S_tot65
        tr = a.tau_t / a.rho_escenario
        Ae = y * J / (N * S * tr)
    elif version == "p1":
        y, N, J, S = a.TBP_C_p1, a.N_t, a.J65, a.S_tot65
        Ae = a.A_eph
        tr = y * J / (N * S * Ae)
    else:
        raise ValueError("version debe ser 'p4' o 'p1'")
    c = pd.DataFrame({"lnTBP": np.log(y), "N": np.log(N), "S65": np.log(S), "Abar": np.log(Ae),
                      "tau_rho": np.log(tr), "J": -np.log(J)})
    c["TBP"] = y.values
    c.index.name = "cohorte"
    assert np.allclose(c[["N", "S65", "Abar", "tau_rho", "J"]].sum(axis=1), c.lnTBP, atol=1e-9), "descomposición no cierra"
    return c


def _pend(y, x):
    return float(np.polyfit(x, y, 1)[0])


FACTORES = ["N", "S65", "Abar", "tau_rho", "J"]


def ventanas_moviles(comp, ancho=20):
    """Para cada ventana de `ancho` cohortes consecutivas: media de TBP, pendiente de ln TBP por cohorte (en %/cohorte) y
    contribución de cada factor (motor_baja = el que más empuja hacia abajo; motor_sube = el que más empuja hacia arriba). Como la pendiente MCO es lineal, la suma de las pendientes de los factores = pendiente
    de ln TBP (exacto). También se da el cambio extremo a extremo (Δln) por factor."""
    filas = []
    co = comp.index.values
    for i in range(0, len(co) - ancho + 1):
        w = comp.iloc[i:i + ancho]
        x = np.arange(ancho, dtype=float)
        r = dict(ini=int(co[i]), fin=int(co[i + ancho - 1]), media_TBP=float(w.TBP.mean()),
                 pend_lnTBP_pct=100 * _pend(w.lnTBP.values, x))
        for f in FACTORES:
            r[f"pend_{f}_pp"] = 100 * _pend(w[f].values, x)
            r[f"dln_{f}_pp"] = 100 * float(w[f].iloc[-1] - w[f].iloc[0])
        r["dln_total_pp"] = 100 * float(w.lnTBP.iloc[-1] - w.lnTBP.iloc[0])
        r["control_suma_pend"] = sum(r[f"pend_{f}_pp"] for f in FACTORES) - r["pend_lnTBP_pct"]
        # qué factor empuja más hacia abajo / hacia arriba en la ventana (mayor pendiente negativa / positiva)
        r["motor_baja"] = min(FACTORES, key=lambda ff: r[f"pend_{ff}_pp"])
        r["motor_sube"] = max(FACTORES, key=lambda ff: r[f"pend_{ff}_pp"])
        filas.append(r)
    return pd.DataFrame(filas)


def _rss(y, X):
    b = np.linalg.lstsq(X, y, rcond=None)[0]
    e = y - X @ b
    return float(e @ e), b


def chow_f(y, x, corte, k=2):
    """F de Chow. k=2: constante+tendencia en cada tramo (x<corte vs x>=corte); k=1: solo media. Descriptivo."""
    y = np.asarray(y, float); x = np.asarray(x, float)
    X = np.column_stack([np.ones(len(x)), x])[:, :k]
    m = x < corte
    if m.sum() <= k or (~m).sum() <= k:
        return np.nan
    rp, _ = _rss(y, X)
    r1, _ = _rss(y[m], X[m]); r2, _ = _rss(y[~m], X[~m])
    return ((rp - (r1 + r2)) / k) / ((r1 + r2) / (len(y) - 2 * k))


def sup_f(y, x, cortes, k=2):
    """F de Chow para cada corte candidato; devuelve (Serie F por corte, corte del máximo, F máximo)."""
    F = pd.Series({int(c): chow_f(y, x, c, k) for c in cortes})
    return F, int(F.idxmax()), float(F.max())


def wald_hac(y, x, corte, lag=None):
    """Wald (2 restricciones: salto de nivel y cambio de pendiente en `corte`) con covarianza Newey-West (Bartlett).
    y = b0 + b1·t + b2·D + b3·D·(t−corte) + e,  D = 1[t>=corte]."""
    y = np.asarray(y, float); x = np.asarray(x, float); n = len(y)
    D = (x >= corte).astype(float)
    X = np.column_stack([np.ones(n), x, D, D * (x - corte)])
    b = np.linalg.lstsq(X, y, rcond=None)[0]
    e = y - X @ b
    L = int(np.floor(4 * (n / 100) ** (2 / 9))) if lag is None else lag
    Xe = X * e[:, None]
    S = Xe.T @ Xe
    for l in range(1, L + 1):
        G = Xe[l:].T @ Xe[:-l]
        S += (1 - l / (L + 1)) * (G + G.T)
    XtXi = np.linalg.inv(X.T @ X)
    V = XtXi @ S @ XtXi
    R = np.array([[0, 0, 1, 0], [0, 0, 0, 1]], float)
    rb = R @ b
    return float(rb @ np.linalg.inv(R @ V @ R.T) @ rb)


def sup_wald_hac(y, x, cortes, lag=None):
    W = pd.Series({int(c): wald_hac(y, x, c, lag) for c in cortes})
    return W, int(W.idxmax()), float(W.max())


def permutacion_bloques(y, x, cortes, estadistico="F", bloque=8, B=499, seed=1):
    """P-valor por permutación CIRCULAR por bloques del sup-estadístico. Nula: una tendencia lineal única + ruido con la
    autocorrelación local preservada (se reordenan bloques de `bloque` residuos). SALVEDAD: con una serie determinística,
    suave y con ciclos largos el 'ruido' no es ruido; este p-valor es una vara de comparación, no una prueba formal."""
    rng = np.random.default_rng(seed)
    y = np.asarray(y, float); x = np.asarray(x, float); n = len(y)
    X = np.column_stack([np.ones(n), x])
    b = np.linalg.lstsq(X, y, rcond=None)[0]
    fit = X @ b; res = y - fit
    f = (lambda yy: sup_f(yy, x, cortes)[2]) if estadistico == "F" else (lambda yy: sup_wald_hac(yy, x, cortes)[2])
    obs = f(y)
    nb = int(np.ceil(n / bloque))
    sup = np.empty(B)
    for i in range(B):
        ini = rng.permutation(n)[:nb] if nb <= n else rng.integers(0, n, nb)
        idx = np.concatenate([(np.arange(bloque) + s0) % n for s0 in ini])[:n]
        sup[i] = f(fit + res[idx])
    p = (1 + int((sup >= obs).sum())) / (B + 1)
    return obs, p, sup


def tabla_quiebre(serie, cortes=range(2000, 2019), nombre="TBP_C_p4", bloques=(5, 15), B=499):
    """Tabla resumen: F por corte, sup-F y su argmax, Wald-HAC y su argmax, p-valores por bloques, y el F del corte 2011."""
    y = serie.values.astype(float); x = serie.index.values.astype(float)
    F, cF, fmax = sup_f(y, x, cortes)
    W, cW, wmax = sup_wald_hac(y, x, cortes)
    out = dict(serie=nombre, corte_supF=cF, supF=fmax, F_2011=float(F.get(2011, np.nan)), corte_supWaldHAC=cW, supWaldHAC=wmax)
    for bl in bloques:
        _, pF, _ = permutacion_bloques(y, x, cortes, "F", bl, B)
        _, pW, _ = permutacion_bloques(y, x, cortes, "W", bl, B)
        out[f"p_perm_supF_bloque{bl}"] = pF
        out[f"p_perm_supWald_bloque{bl}"] = pW
    por_corte = pd.DataFrame({"corte": list(F.index), "F_chow": F.values, "Wald_HAC": W.values})
    por_corte["serie"] = nombre
    return out, por_corte


# ==============================================================================================
# C) AUTOTEST (SINTÉTICO)
# ==============================================================================================
def _banner(t):
    print("=" * 78); print(t); print("=" * 78)


def autotest(verbose=True):
    """Pruebas con datos SINTÉTICOS (no son Argentina)."""
    if verbose:
        _banner("AUTOTEST tbp_estabilidad — DATOS SINTÉTICOS (no son Argentina)")
    # 1) error_pct e interpolación
    assert abs(float(error_pct(110, 100)) - 10.0) < 1e-12 and abs(float(error_pct(90, 100)) + 10.0) < 1e-12
    serie = {2021: 100.0, 2022: 200.0}
    assert abs(interpolar_a_fecha(serie, _dt.date(2021, 7, 1)) - 100.0) < 1e-9
    assert abs(interpolar_a_fecha(serie, _dt.date(2022, 1, 1)) - (100 + 100 * 184 / 365)) < 1e-9
    # 2) descomposición exacta: construir un av sintético con factores conocidos
    co = np.arange(1950, 2036)
    rng = np.random.default_rng(0)
    N = 5e5 * np.exp(0.01 * (co - 1950) - 0.0002 * (co - 1950) ** 2) * np.exp(rng.normal(0, .01, len(co)))
    S = np.linspace(0.7, 0.93, len(co)); Ab = 14 + np.sin(co / 9.0); J = 4e6 * np.exp(0.03 * (co - 1950) ** 0.8)
    tau = 0.2177; rho = np.where(co < 1960, 0.4, 0.403)
    tbp = N * S * Ab * tau / (rho * J)
    av = pd.DataFrame(dict(cohorte=co, N_4=N, N_t=N, S_tot65=S, J_star_4=J, J65=J, A_eph=Ab, tau_t=tau,
                           rho_escenario=rho, TBP_C_p4=tbp, TBP_C_p1=N * S * Ab * (tau / 0.403) / J))
    c4 = componentes_log(av, "p4"); c1 = componentes_log(av, "p1")
    assert np.allclose(c4.Abar, np.log(Ab[:75])) and np.allclose(c4.N, np.log(N[:75]))
    assert np.allclose(c1.tau_rho, np.log(tau / 0.403))
    # 3) ventanas: la suma de pendientes cierra y el número de ventanas es n−ancho+1
    v = ventanas_moviles(c4, 20)
    assert len(v) == 75 - 20 + 1 and np.allclose(v.control_suma_pend, 0, atol=1e-9)
    # 4) sup-F recupera un quiebre sintético (serie con quiebre de pendiente en 2005 + ruido AR(1))
    x = np.arange(1950, 2025, dtype=float)
    e = np.zeros(len(x))
    for i in range(1, len(x)):
        e[i] = 0.5 * e[i - 1] + rng.normal(0, 0.01)
    y_q = 0.5 + 0.0 * x - 0.02 * np.clip(x - 2005, 0, None) + e
    F, cF, fm = sup_f(y_q, x, range(2000, 2019))
    assert abs(cF - 2005) <= 3, cF
    W, cW, wm = sup_wald_hac(y_q, x, range(2000, 2019))
    assert abs(cW - 2005) <= 4, cW
    # 5) bajo la nula (tendencia lineal + AR) el p-valor por bloques NO debe ser sistemáticamente ínfimo
    y_n = 0.5 - 0.002 * (x - 1950) + e
    _, p_n, _ = permutacion_bloques(y_n, x, range(2000, 2019), "F", 8, B=199, seed=3)
    _, p_q, _ = permutacion_bloques(y_q, x, range(2000, 2019), "F", 8, B=199, seed=3)
    assert p_q < 0.05 and p_n > p_q
    # 6) sigma por horizonte y Monte Carlo (reproducibilidad y monotonía en σ)
    assert abs(sigma_por_horizonte(0.02, 9, 36) - 0.04) < 1e-12 and sigma_por_horizonte(0.02, 9, 36, "constante") == 0.02
    _, a = mc_tbp(4e5, 0.9, 1.5e7, 0.015, 0.01, n=2000); _, a2 = mc_tbp(4e5, 0.9, 1.5e7, 0.015, 0.01, n=2000)
    _, b = mc_tbp(4e5, 0.9, 1.5e7, 0.15, 0.05, n=2000)
    assert a == a2 and (b["p95"] - b["p5"]) > (a["p95"] - a["p5"])
    # 7) tamaño de pendientes verificables
    assert 15 < tamano_pendientes() < 30
    if verbose:
        print(f"OK: error %, interpolación, descomposición exacta, ventanas (sumas cierran), sup-F (corte hallado {cF}; Wald-HAC {cW}),")
        print(f"    permutación por bloques (p con quiebre {p_q:.3f} < p sin quiebre {p_n:.3f}), σ por horizonte, Monte Carlo.")
        _banner("FIN AUTOTEST — estos números NO son Argentina")
    return True

autotest_estabilidad = autotest
if _autotest_previo_19 is not None:
    autotest = _autotest_previo_19

### 20.3 Autotest con datos sintéticos

Verifica con series **inventadas** (rotuladas como tales): error %, interpolación a fecha censal, descomposición logarítmica exacta, ventanas (las sumas cierran), que el sup-F recupera un quiebre plantado en 2005 y que la permutación por bloques da un $p$ menor con quiebre que sin él, y reproducibilidad del Monte Carlo.

In [ ]:
_tabla_sintetica_19 = autotest_estabilidad()

### 20.4 Carga de datos

Se leen de las rutas candidatas `../datos/procesados`, `datos/procesados`, `../datos/crudos`, `datos/crudos`: el modelo avanzado (`tbp_modelo_avanzado_1950_2035.csv`) y el xls de INDEC 2013 (población total al 1-jul, 2010-2040). Las series de la ONU y los nacimientos DEIS salen de `s` (sección 0). Las cifras del Censo 2022 y de INDEC 2025 están transcriptas en el módulo con su página.

In [ ]:
import os, numpy as np, pandas as pd, matplotlib.pyplot as plt
try: display
except NameError: display = print
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 30)

def buscar_archivo_19(nombre):
    for c in ["../datos/procesados", "datos/procesados", "../datos/crudos", "datos/crudos", DATA]:
        p = os.path.join(c, nombre)
        if os.path.exists(p):
            return p
    return None

DEST19 = next((p for p in ["../datos/procesados", "datos/procesados"] if os.path.isdir(p)), OUT)
os.makedirs(OUT, exist_ok=True)

# `s` (series ONU/DEIS) puede haber sido pisada por una variable de bucle en secciones previas (p. ej. 16.2): se valida y, si hace falta, se recarga.
S19 = s if (isinstance(s, pd.DataFrame) and "deis_nacimientos" in s.columns) else pd.read_csv(buscar_archivo_19("series_tbp_extraidas.csv")).set_index("year")

f_av = buscar_archivo_19("tbp_modelo_avanzado_1950_2035.csv")
HAY_AV19 = f_av is not None
AV19 = pd.read_csv(f_av) if HAY_AV19 else None
print("Modelo avanzado:", f_av if HAY_AV19 else "NO ENCONTRADO (las partes B y C se omiten)")

# INDEC 2013 (Cuadro 1: población total al 1-jul 2010-2040) — archivo local del proyecto
X13 = None
f_x13 = buscar_archivo_19("indec_c1_proyecciones_nac_2010_2040.xls")
try:
    _x = pd.read_excel(f_x13, header=None).iloc[:, :4].dropna()
    _x = _x[pd.to_numeric(_x[0], errors="coerce").between(2010, 2040)]
    X13 = pd.Series(_x[1].astype(float).values, index=_x[0].astype(int).values)
    assert len(X13) == 31 and X13[2010] > 4e7
except Exception as e:
    X13 = None; print("No se pudo leer el xls de INDEC 2013 (", type(e).__name__, "): se omiten esas filas.")
print("INDEC 2013 leído:", X13 is not None, "| S19 (series ONU/DEIS):", S19.shape)

### 20.5 Backtest de insumos (A)

**Qué se compara y con qué honestidad:**

* **Nacimientos, ONU WPP 2024 (jul-2024) vs DEIS.** 2022: +0,2 % (dato ya incorporado); **2023: +9,3 %; 2024: +22,6 %**. La ONU sobrestimó porque la natalidad siguió cayendo. Salvedad: el DEIS 2024 puede ser provisorio (las inscripciones tardías lo suben), así que el error de 2024 es una cota alta. El proxy con la población de 0 años de INDEC 2025 da +3,7 % sobre el promedio DEIS 2023-24, coherente con eso.
* **Población total, INDEC 2013 vs Censo 2022 (a 12 años).** +0,6 % contra el censo enumerado (45,89 M, p. 12 del PDF de INDEC 2025) y +0,1 % contra la estimación INDEC a la fecha censal (46,12 M, p. 19). Acertó el total, pero eso **no prueba** que haya acertado nacimientos y 65+ por separado.
* **Revisión INDEC 2013 → INDEC 2025.** Las proyecciones se separan: +2,3 % (2025) a +11,1 % (2040). Es la señal de cuánto pesa un escenario de fecundidad más baja; no es error contra datos.
* **65+ (J).** La ONU WPP 2024 da 65+ un 0,8 % por debajo del Censo 2022 (WPP 2024 **no** incorpora el censo: su población total 2022 queda 1,1 % abajo). Contra INDEC 2025 la diferencia va de −0,7 % a −2,5 % (RMSE 1,5 %): es la **incertidumbre mínima** de $J$ entre dos fuentes serias.

**Idea clave:** $J$ de las cohortes ≤ 2024 es la población 65+ de 2076-2089, formada por gente **ya nacida**; su incertidumbre viene de mortalidad y migración, no de nacimientos futuros. El error de $N$ afecta a las cohortes ≥ 2025.

In [ ]:
BT19 = backtest_tabla(S19, X13)
cols = ["insumo", "proyeccion", "referencia", "horizonte_anios", "proyectado", "observado", "error_pct", "tipo"]
with pd.option_context("display.float_format", lambda v: f"{v:,.1f}", "display.max_colwidth", 70):
    display(BT19[BT19.bloque != "A4 pendiente"][cols].reset_index(drop=True))

# Gráfico: errores % (positivo = la proyección sobrestimó)
SEL19 = ["Nacimientos 2022", "Nacimientos 2023", "Nacimientos 2024",
         "Población total al 18-may-2022", "Población total 2030: INDEC 2013 vs INDEC 2025",
         "Población total 2040: INDEC 2013 vs INDEC 2025", "65+ al 18-may-2022", "65+ 2030: ONU WPP 2024 vs INDEC 2025",
         "65+ 2040: ONU WPP 2024 vs INDEC 2025"]
g19 = BT19[BT19.insumo.isin(SEL19) & (BT19.proyeccion != "(vintage anterior)")].drop_duplicates("insumo").set_index("insumo").loc[SEL19]
COL_T = {"backtest": "#0072b2", "revisi": "#d55e00", "dispersi": "#009e73", "chequeo": "#999999"}
def _col(t):
    return next((c for k, c in COL_T.items() if t.startswith(k) or k in t), "#999999")
fig, ax = plt.subplots(figsize=(10, 4.8))
yy = np.arange(len(g19))[::-1]
ax.barh(yy, g19.error_pct, color=[_col(tp) for tp in g19.tipo])
for yi, v19 in zip(yy, g19.error_pct):
    ax.text(v19 + (0.6 if v19 >= 0 else -0.6), yi, f"{v19:+.1f} %", va="center", ha="left" if v19 >= 0 else "right", fontsize=9)
ax.set_yticks(yy); ax.set_yticklabels([f"{i}\n({r19.proyeccion})" for i, r19 in g19.iterrows()], fontsize=8)
ax.axvline(0, color="k", lw=0.8); ax.set_xlim(-6, 30)
ax.set_xlabel("Error % = 100·(proyectado/observado − 1);  > 0: la proyección sobrestimó")
ax.set_title("Sección 19 — Errores de proyecciones pasadas de los insumos", fontweight="bold")
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color="#0072b2", label="backtest legítimo"), Patch(color="#d55e00", label="revisión entre vintages (no es error vs hechos)"),
                   Patch(color="#009e73", label="dispersión entre fuentes"), Patch(color="#999999", label="chequeo de nivel")], loc="lower right", fontsize=8)
plt.tight_layout(); plt.savefig(os.path.join(OUT, "seccion_20_backtest_insumos.png"), dpi=130); plt.show()

e_nac = BT19[(BT19.insumo.isin(["Nacimientos 2023", "Nacimientos 2024"]))].error_pct.values
RMSE_N19 = float(np.sqrt(np.mean(e_nac ** 2)))
e_J19, RMSE_J19 = dispersion_J(S19.un_pob_65mas_miles)
print(f"Nacimientos ONU WPP 2024 vs DEIS: 2023 {e_nac[0]:+.1f} %, 2024 {e_nac[1]:+.1f} %  → RMSE (n=2) = {RMSE_N19:.1f} %")
print(f"65+ ONU WPP 2024 vs INDEC 2025 (2022-2040): RMSE = {RMSE_J19:.2f} %; rango {e_J19.min():+.2f} a {e_J19.max():+.2f} %")

### 20.6 De los errores a una banda de incertidumbre para $N$ y $J$ (reemplaza el ±1,5 % / ±1 %)

**Propuesta.** En el Monte Carlo de 16.4, $N\sim\mathcal N(\hat N,\,0{,}015\hat N)$ y $J\sim\mathcal N(\hat J,\,0{,}01\hat J)$ eran valores *arbitrarios*. Se reemplazan por desvíos medidos:

* $\sigma_J$ = RMSE de la diferencia ONU–INDEC en 65+ (≈ 1,5 %), y una variante que **crece con la raíz del horizonte** (supuesto *heurístico*, no estimado: $\sigma(h)=\sigma_{ref}\sqrt{h/h_{ref}}$ con $h_{ref}=9$).
* $\sigma_N$ **solo para cohortes ≥ 2025** (N proyectado): RMSE del backtest de nacimientos 2023-24 (≈ 17 %). Es una referencia *empírica pero débil* (2 puntos, un régimen de caída, sesgo de un solo signo). La cohorte 2024 usa dato DEIS y conserva 1,5 %.

La misma semilla y sorteos que 16.4 → el escenario A reproduce exactamente el P5 = 0,152 / P95 = 0,269 de `monte_carlo_resumen_2024.csv`.

In [ ]:
if HAY_AV19:
    a19 = AV19.set_index("cohorte")
    H_REF = 9      # distancia media (años) de 2022..2040 al año base 2022-2031: referencia del RMSE de J (HEURÍSTICA)
    filas_mc = []
    sims = {}
    for coh in (2024, 2030):
        r19 = a19.loc[coh]; hJ = int(r19.anio_65 - 2024)
        esc = [("A. original (σN=1,5 %, σJ=1 %)", 0.015, 0.01),
               ("B. σJ empírica (RMSE entre fuentes)", 0.015 if coh == 2024 else RMSE_N19 / 100, RMSE_J19 / 100),
               ("C. B + σJ creciente con √horizonte", 0.015 if coh == 2024 else RMSE_N19 / 100,
                sigma_por_horizonte(RMSE_J19 / 100, H_REF, hJ))]
        for nom, sN, sJ in esc:
            sim, q19 = mc_tbp(r19.N_4, r19.S_tot65, r19.J_star_4, sN, sJ)
            sims[(coh, nom[0])] = sim
            filas_mc.append(dict(cohorte=coh, escenario=nom, sigma_N_pct=100 * sN, sigma_J_pct=100 * sJ, p5=q19["p5"], p50=q19["p50"], p95=q19["p95"],
                                 ancho_90=q19["p95"] - q19["p5"], base_punto=r19.TBP_C_p4, ref_cohorte_2000=a19.loc[2000, "TBP_C_p4"],
                                 prob_supera_2000=float((sim >= a19.loc[2000, "TBP_C_p4"]).mean())))
    MC19 = pd.DataFrame(filas_mc)
    with pd.option_context("display.float_format", lambda v: f"{v:,.3f}"):
        display(MC19)
    print("Nota: N de la cohorte 2024 es dato DEIS (posiblemente provisorio); N de 2030 es escenario → σN de 2030 usa el RMSE del backtest (n=2, régimen de caída).")

### 20.7 Estabilidad por ventanas móviles de 20 cohortes (B)

Para cada ventana: media de $TBP_C$, pendiente de $\ln TBP$ por cohorte y contribución de cada factor con la descomposición logarítmica **exacta**
$$\ln TBP_C=\ln N+\ln(1-\mu)+\ln\bar A+\ln(\tau/\rho)-\ln J$$
(la suma de las pendientes de los factores es la pendiente del total; el código lo verifica). En `p4`, $\bar A$ es el **efectivo** que cierra la identidad (incluye el reparto por sexo y edad de retiro), $J=J^*$ y $\tau/\rho$ varía solo por el $\rho$ de las primeras cohortes; en `p1`, $\bar A$ es el de la EPH y $J$ = 65+.

**Lectura (cifras del modelo avanzado, cohortes 1950-2024):**

* En **todas las ventanas que terminan hasta 2020**, el factor que más empuja hacia abajo es $J$ (el envejecimiento: −1,7 a −2,1 pp por cohorte en 1950-1989, −0,35 en la última). $N$ **empuja hacia arriba** (hasta +2,2 pp en la ventana 1960-1979) y recién se vuelve negativo cuando entran las cohortes post-2015.
* **El motor pasa a ser $N$ recién en las ventanas que terminan en 2021-2024** (2005-2024: $N$ = −2,5 pp por cohorte, $J$ = −0,35). **No** se verifica que "desde ~2000 domine $N$": en 2000-2019 $N$ prácticamente no aporta (+0,06).
* En el medio (ventanas 1974-2005) $\bar A$ es el segundo freno (hasta −1,3 pp), por el patrón de aportes de la EPH.
* La media de la ventana baja de 0,46 (1970-1989) a 0,30 (2005-2024).

In [ ]:
if HAY_AV19:
    COMP4 = componentes_log(AV19, "p4", hasta=2024); COMP1 = componentes_log(AV19, "p1", hasta=2024)
    V4 = ventanas_moviles(COMP4, 20); V1 = ventanas_moviles(COMP1, 20)
    assert np.allclose(V4.control_suma_pend, 0, atol=1e-8) and np.allclose(V1.control_suma_pend, 0, atol=1e-8)
    print("Control: en cada ventana, la suma de pendientes de N, S65, Ā, τ/ρ y J = pendiente de ln TBP (exacto).")
    pick = [1950, 1960, 1970, 1980, 1990, 2000, 2005]
    t4 = V4[V4.ini.isin(pick)].set_index("ini")
    with pd.option_context("display.float_format", lambda v: f"{v:,.2f}"):
        display(t4[["fin", "media_TBP", "pend_lnTBP_pct", "pend_N_pp", "pend_S65_pp", "pend_Abar_pp", "pend_tau_rho_pp", "pend_J_pp", "motor_baja", "motor_sube"]]
                .rename(columns=lambda c: c.replace("pend_", "").replace("_pp", " (pp/coh)")))
    print("Unidades: puntos porcentuales de ln TBP por cohorte (≈ % por cohorte). 'J' ya viene con signo −ln J (J que crece resta).")

    fig, ax = plt.subplots(1, 3, figsize=(18, 4.8))
    COLF = {"N": "#d55e00", "S65": "#009e73", "Abar": "#0072b2", "tau_rho": "#999999", "J": "#cc79a7"}
    ETQF = {"N": "N (nacimientos)", "S65": "S65 = 1−μ", "Abar": "Ā (efectivo)", "tau_rho": "τ/ρ", "J": "−ln J (jubilados)"}
    for a_, V, tit in ((ax[0], V4, "TBP_C_p4 (J*, sexo/edad de retiro)"), (ax[1], V1, "TBP_C_p1 (J = 65+)")):
        pos = np.zeros(len(V)); neg = np.zeros(len(V))
        for ff in FACTORES:
            vv = V[f"pend_{ff}_pp"].values
            base = np.where(vv >= 0, pos, neg)
            a_.bar(V.fin, vv, bottom=base, color=COLF[ff], label=ETQF[ff], width=0.9)
            pos += np.where(vv >= 0, vv, 0); neg += np.where(vv < 0, vv, 0)
        a_.plot(V.fin, V.pend_lnTBP_pct, "k-", lw=2, label="pendiente de ln TBP")
        a_.axhline(0, color="k", lw=0.6); a_.set_title(f"Motor por ventana de 20 cohortes\n{tit}", fontsize=10, fontweight="bold")
        a_.set_xlabel("última cohorte de la ventana"); a_.set_ylabel("pp de ln TBP por cohorte")
    ax[0].legend(fontsize=7.5, loc="lower left")
    ax[2].plot(V4.fin, V4.media_TBP, color="#0072b2", lw=2, label="p4: media de la ventana")
    ax[2].plot(V1.fin, V1.media_TBP, color="#d55e00", lw=2, label="p1: media de la ventana")
    ax[2].set_title("Media de TBP_C por ventana", fontsize=10, fontweight="bold"); ax[2].set_xlabel("última cohorte de la ventana"); ax[2].set_ylabel("años"); ax[2].legend(fontsize=8)
    plt.tight_layout(); plt.savefig(os.path.join(OUT, "seccion_20_estabilidad_ventanas.png"), dpi=130); plt.show()

### 20.8 Test de quiebre: sup-F en lugar de un único corte (B)

Se calcula el F de Chow (constante + tendencia) para **cada corte candidato 2000-2018** sobre las cohortes 1950-2024 y se reporta el máximo (sup-F). También un Wald con covarianza **Newey-West (HAC)** y, como versión robusta, **permutación circular por bloques** de los residuos de una tendencia lineal (bloques de 5 y 15 cohortes).

**Aviso:** la serie es determinística, suave y autocorrelada. Estos estadísticos **describen** dónde cambia la pendiente; los $p$ no tienen la interpretación inferencial habitual. La permutación por bloques es solo una vara de comparación (su "ruido" son residuos de una curva que no es lineal).

**Resultados:**

* **El máximo está en 2015** (TBP_C_p4: F = 9,37; TBP_C_p1: 9,10; nacimientos: 168). El F del corte 2011 es 8,40 (se reproduce exactamente el 8,3972 del test existente). La curva es una meseta entre 2013 y 2018 (F de 9,1 a 9,4): **el corte 2011 no es muy distinto, pero tampoco es el "natural"**; el cambio de pendiente de $TBP$ sigue al de $N$ (pico 2014, caída desde 2015).
* **Robustez:** con permutación por bloques, el $p$ del sup-F de $TBP$ es 0,43-0,62 (no distinguible de una tendencia lineal con ruido autocorrelado), mientras que para $N$ es ≈ 0,003. El Wald-HAC da $p$ = 0,005-0,033 (p4) y 0,010-0,060 (p1) según el bloque. **Los tests no coinciden entre sí**: la conclusión no es robusta como "quiebre estadístico".
* **Conclusión que sale:** la pendiente de $TBP$ cambia alrededor de 2014-2015 y lo hace por $N$. **Conclusión que NO sale:** que exista un "quiebre estructural significativo" en 2011 (ni en ninguna fecha) en sentido inferencial.

In [ ]:
if HAY_AV19:
    ser = AV19.set_index("cohorte").loc[:2024]
    CORTES = range(2000, 2019)
    series_q = {"TBP_C_p4": ser.TBP_C_p4, "TBP_C_p1": ser.TBP_C_p1, "N_t (nacimientos)": ser.N_t.astype(float)}
    res_q, por_corte = [], []
    for nom, sr in series_q.items():
        o, pc = tabla_quiebre(sr, CORTES, nom, bloques=(5, 15), B=399)
        res_q.append(o); por_corte.append(pc)
    QB19 = pd.DataFrame(res_q); PC19 = pd.concat(por_corte, ignore_index=True)
    with pd.option_context("display.float_format", lambda v: f"{v:,.3f}"):
        display(QB19.set_index("serie").T)
    f11 = float(PC19[(PC19.serie == "TBP_C_p4") & (PC19.corte == 2011)].F_chow.iloc[0])
    print(f"Chow en 2011 (TBP_C_p4, k=2) recalculado aquí: F = {f11:.4f}  (el test existente informa 8,3972) ", "→ coincide" if abs(f11 - 8.3972) < 0.01 else "→ NO coincide, revisar")
    fig, ax = plt.subplots(1, 2, figsize=(14, 4.6))
    p4 = PC19[PC19.serie == "TBP_C_p4"]
    ax[0].plot(p4.corte, p4.F_chow, "o-", color="#0072b2", label="F de Chow (tendencia) por corte")
    ax[0].axvline(2011, color="k", ls="--", lw=1, label="corte del test existente (2011)")
    ax[0].axvline(QB19.set_index("serie").loc["TBP_C_p4", "corte_supF"], color="#d55e00", ls=":", lw=2, label="máximo (sup-F)")
    ax[0].set_xticks(list(CORTES)[::2]); ax[0].set_xlabel("corte candidato (primera cohorte del 2.º tramo)"); ax[0].set_ylabel("F"); ax[0].legend(fontsize=8)
    ax[0].set_title("TBP_C_p4: el estadístico casi no distingue entre cortes 2013-2018", fontsize=10, fontweight="bold")
    for nom, col in (("TBP_C_p4", "#0072b2"), ("N_t (nacimientos)", "#d55e00")):
        q19 = PC19[PC19.serie == nom]; ax[1].plot(q19.corte, q19.F_chow / q19.F_chow.max(), "o-", color=col, label=nom + " (F / F máx.)")
    ax[1].axvline(2011, color="k", ls="--", lw=1); ax[1].axvline(2014, color="grey", ls=":", lw=1.5, label="pico de nacimientos DEIS (2014)")
    ax[1].set_xticks(list(CORTES)[::2]); ax[1].set_xlabel("corte candidato"); ax[1].set_ylabel("F normalizado"); ax[1].legend(fontsize=8)
    ax[1].set_title("El quiebre de TBP sigue al de N", fontsize=10, fontweight="bold")
    plt.tight_layout(); plt.savefig(os.path.join(OUT, "seccion_20_supF.png"), dpi=130); plt.show()
    print("RECORDATORIO: serie determinística y autocorrelada → F, Wald y p-valores son DESCRIPTIVOS; no prueban un 'quiebre estructural' en sentido inferencial.")

### 20.9 Salidas y lista de lo que habría que descargar

Se escriben en `datos/procesados/`: `backtest_insumos.csv` (una fila por comparación; las filas `PENDIENTE DE DESCARGA` son la lista de abajo), `estabilidad_ventanas.csv` (una fila por ventana y versión) y, de apoyo, `estabilidad_quiebre.csv` (F y Wald por corte) y `backtest_bandas_montecarlo.csv`.

**Qué habría que bajar (NO se descargó; tamaños medidos con cabeceras HTTP):**

| Archivo | Tamaño (cabeceras HTTP) | Para qué |
|---|---|---|
| INDEC 2013, Serie Análisis Demográfico n.º 35 (PDF) `https://www.indec.gob.ar/ftp/cuadros/publicaciones/proyeccionesyestimaciones_nac_2010_2040.pdf` | 267.025 B (≈ 0,26 MB) | 65+ y natalidad/TGF proyectados por INDEC en 2013 (Cuadros 2 y 5) → error de J y de N contra el Censo 2022 y DEIS |
| WPP 2019 (paquete R `wpp2019_1.1-1.tar.gz`, CRAN) | 4.684.504 B (≈ 4,7 MB) | vintage 2019: horizonte ~4-5 años para 2023-24 |
| WPP 2017 (`wpp2017_1.2-3.tar.gz`, CRAN) | 4.743.882 B (≈ 4,7 MB) | vintage intermedio |
| WPP 2015 (`Archive/wpp2015/wpp2015_1.1-2.tar.gz`, CRAN) | 4.585.228 B (≈ 4,6 MB) | horizonte ~8-9 años |
| WPP 2012 (`wpp2012_2.2-1.tar.gz`, CRAN) | 4.902.348 B (≈ 4,9 MB) | horizonte ~10-12 años |
| WPP 2010 | no verificado (sin paquete en CRAN; las URL históricas de population.un.org devolvieron 404) | habría que pedirlo al archivo de la División de Población de la ONU |

Total verificable: ≈ 19,2 MB. Las URL del sitio de la ONU para los Excel históricos devolvieron 404, por eso se proponen los paquetes R de CRAN (que redistribuyen las tablas de cada revisión); el contenido exacto de cada paquete hay que confirmarlo al abrirlo.

In [ ]:
os.makedirs(DEST19, exist_ok=True)
BT19.to_csv(os.path.join(DEST19, "backtest_insumos.csv"), index=False, encoding="utf-8")
print("Escrito:", os.path.join(DEST19, "backtest_insumos.csv"), BT19.shape)
if HAY_AV19:
    EST19 = pd.concat([V4.assign(version="p4"), V1.assign(version="p1")], ignore_index=True)
    EST19 = EST19[["version"] + [c for c in EST19.columns if c != "version"]]
    EST19.to_csv(os.path.join(DEST19, "estabilidad_ventanas.csv"), index=False, encoding="utf-8")
    PC19.to_csv(os.path.join(DEST19, "estabilidad_quiebre.csv"), index=False, encoding="utf-8")
    MC19.to_csv(os.path.join(DEST19, "backtest_bandas_montecarlo.csv"), index=False, encoding="utf-8")
    print("Escrito:", os.path.join(DEST19, "estabilidad_ventanas.csv"), EST19.shape, "| estabilidad_quiebre.csv", PC19.shape, "| backtest_bandas_montecarlo.csv", MC19.shape)
print(f"A descargar (solo lista; verificable): ~{tamano_pendientes():.1f} MB en {sum(1 for p in PENDIENTES_DESCARGA if p['bytes'])} archivos + WPP 2010 (no verificado).")

### 20.10 Qué sirve validar y qué no

| Qué se prueba | Qué responde | Datos que usa | Qué NO responde |
|---|---|---|---|
| **Backtest de N**: nacimientos proyectados por la ONU (WPP 2024) vs DEIS 2023-24 | ¿Cuánto se equivocó una proyección real publicada *antes* del dato? (+9,3 % y +22,6 %) | `series_tbp_extraidas.csv` | No dice cuánto se equivocarán las proyecciones 2025+; son solo 2 puntos en un régimen de caída de la fecundidad |
| **Backtest de población total**: INDEC 2013 vs Censo 2022 | ¿Acertaron las proyecciones largas (12 años) el tamaño de la población? (+0,6 %) | xls INDEC 2013 + PDF INDEC 2025 (pp. 12 y 19) | No dice nada de N ni de 65+ por separado (errores de signo opuesto se compensan) |
| **Revisión entre vintages**: INDEC 2013 vs INDEC 2025 (2025-2040) | ¿Cuánto cambia el escenario al incorporar un censo nuevo? (+2,3 % en 2025 a +11,1 % en 2040) | xls INDEC 2013 + PDF INDEC 2025 (pp. 39-41) | No es error contra hechos: la "realidad" es otra proyección |
| **Dispersión de J entre fuentes**: ONU WPP 2024 vs INDEC 2025 (65+) | ¿Cuánta incertidumbre mínima hay en J? (RMSE 1,5 %; −0,7 a −2,5 %) | `series_tbp_extraidas.csv` + PDF INDEC 2025 | No es un error de proyección; las dos fuentes pueden equivocarse juntas |
| **Backtest de vintages anteriores** (WPP 2012/2015/2017/2019, INDEC 2013 por edad) | Cómo crece el error con el horizonte (necesario para una banda seria de N y J) | **PENDIENTE**: ver lista de descargas | Hoy no se puede responder: no hay datos en el proyecto |
| **Ventanas móviles + descomposición logarítmica** | ¿Qué factor (N, S65, Ā, τ/ρ, J) empuja la serie en cada tramo? | `tbp_modelo_avanzado_1950_2035.csv` | No es causal ni predictivo; es contabilidad exacta de lo ya calculado |
| **sup-F / Chow por corte** (y Wald-HAC, permutación por bloques) | ¿Dónde cambia la pendiente de la serie? (máximo en 2015) | ídem | **No** prueba un "quiebre estructural" inferencial: la serie es determinística y autocorrelada |
| **Split IS/OOS 2011** (test existente) | Describe que las cohortes 2011-2024 tienen TBP menor que las anteriores (−28,6 %) | ídem | No es out-of-sample (los insumos de esas cohortes son proyecciones a 2076-2089); el corte 2011 es arbitrario; los p-valores no tienen lectura inferencial |
| **Sensibilidad / tornado / Monte Carlo** (sección 16) | ¿Cuánto cambia el nivel si cambian los supuestos? | parámetros con fuente | No valida que los supuestos sean ciertos; los desvíos de N y J eran arbitrarios (se reemplazan acá) |
| **Asserts de identidad** (secciones 3, 7, 15) | ¿El código calcula bien la fórmula? | las propias series | No prueba que la fórmula describa la realidad |

### 20.11 Lectura y límites

* **Qué se puede decir:** (1) la ONU sobrestimó los nacimientos 2023-24 en 9 % y 23 %; (2) INDEC 2013 acertó el total de población a 12 años pero su escenario hoy está 2-11 % arriba del de INDEC 2025; (3) $J$ difiere ~1,5 % entre ONU e INDEC; (4) la caída de $TBP_C$ en las cohortes recientes viene de $N$, mientras que en el resto de la serie el freno principal fue $J$; (5) la pendiente cambia hacia 2014-2015.
* **Qué NO se puede decir:** que el índice "predice" algo; que el tramo 2011-2024 sea out-of-sample; que haya un quiebre estadísticamente significativo; que las bandas de 19.6 sean definitivas (σ de $N$ con 2 observaciones; escalamiento por horizonte es un supuesto).
* **Límites:** los vintages previos (WPP 2012-2019, INDEC 2013 por edad) no están descargados, así que falta medir cómo crece el error con el horizonte; el DEIS 2024 puede ser provisorio; el backtest de 65+ contra el censo usa la ONU, no una proyección de INDEC de la época.
* **Observación sobre 16.5:** con los parámetros de 16.4 se reproduce P95 = 0,269 y la probabilidad de superar la cohorte 2000 (0,312) es ≈ 0,3 % (30 de 10.000), no "< 0,1 %"; la conclusión cualitativa no cambia.